In [10]:
"""
02_train_model.py
=================
Trains the 1D CNN + BiLSTM model on BIM keypoint sequences.

AUGMENTATION STRATEGIES (updated):
  - Spatial      : scale, rotate, translate (existing)
  - Noise        : gaussian noise on keypoints (existing)
  - Mirror       : horizontal flip swapping left/right hand (ENABLED)
  - Hand scale   : independent per-hand scale variation (NEW)
  - Body shift   : per-body-part translation offset (NEW)
  - Perspective  : x-shear to simulate camera angle (NEW)
  - Time warp    : stretch/compress sequence speed (NEW)
  - Frame interp : smooth interpolation instead of frame repeat (NEW)
  - Keypoint drop: random zero-out of landmarks for occlusion (NEW)
  - Mixup        : blend two same-class sequences (NEW)

FIXES CARRIED OVER:
  #3  — input_shape (not batch_input_shape) for TFLite export
  #8  — MIN_NONZERO = 0.45
  #9  — Augmentation AFTER train/test split (no data leakage)
  #10 — EarlyStopping(restore_best_weights=True), no ModelCheckpoint
"""

import os
import json
import math
import pickle
import random
from collections import Counter, defaultdict

import numpy as np
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Bidirectional, BatchNormalization, Conv1D,
    Dense, Dropout, LSTM, MaxPooling1D
)
from tensorflow.keras.models import Sequential
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ==============================
# CONFIG — single source of truth
# ==============================
DATA_PATH       = "dataset_phone"
SEQUENCE_LENGTH = 8
HAND_FEATURES   = 126
FACE_FEATURES   = 60
POSE_FEATURES   = 18
FEATURE_SIZE    = HAND_FEATURES + FACE_FEATURES + POSE_FEATURES  # 204
MIN_NONZERO     = 0.45

TEST_SIZE       = 0.2
BATCH_SIZE      = 32
EPOCHS          = 80

MODEL_SAVE_PATH = "bim_model/handface_pose_cnn_lstm.h5"
TFLITE_PATH     = "bim_model/handface_pose_cnn_lstm.tflite"
LABEL_PATH      = "bim_model/labels.json"
ENCODER_PATH    = "bim_model/label_encoder.pkl"
HISTORY_PATH    = "bim_model/train_history.pkl"

# ── Augmentation config ────────────────────────────────────────────────────
AUGMENTATION_FACTOR = 5        # augmented copies per original sequence

# Spatial
AUG_SCALE_RANGE   = (0.85, 1.15)
AUG_ROT_RANGE     = (-10, 10)
AUG_TRANS_RANGE   = (-0.05, 0.05)

# Noise
AUG_NOISE_STD     = 0.006

# Hand scale (independent per hand)
AUG_HAND_SCALE    = (0.80, 1.20)

# Perspective shear
AUG_SHEAR_RANGE   = (-0.12, 0.12)

# Time warp
AUG_WARP_RANGE    = (0.7, 1.3)
AUG_WARP_PROB     = 0.4        # applied 40% of the time

# Keypoint dropout
AUG_KP_DROP_P     = 0.12       # probability of dropping each body part/point

# Mixup
AUG_MIXUP_ALPHA   = 0.2
AUG_MIXUP_PROB    = 0.3        # applied 30% of the time

# Mirror
AUG_MIRROR_PROB   = 0.5        # applied 50% of the time

os.makedirs("bim_model", exist_ok=True)

# ==============================
# REPRODUCIBILITY
# ==============================
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# ==============================
# NORMALIZATION (identical to server)
# ==============================
def normalize_hand(frame):
    frame = frame.copy()
    xs, ys = frame[0::3], frame[1::3]
    valid = xs != 0
    if not valid.any():
        return frame
    min_x, max_x = xs[valid].min(), xs[valid].max()
    min_y, max_y = ys[valid].min(), ys[valid].max()
    scale = max(max_x - min_x, max_y - min_y)
    if scale == 0: scale = 1
    for i in range(0, len(frame), 3):
        frame[i]     = (frame[i]     - min_x) / scale
        frame[i + 1] = (frame[i + 1] - min_y) / scale
    return frame

def normalize_hand_pair(h):
    return np.concatenate([normalize_hand(h[:63]), normalize_hand(h[63:])])

def normalize_face(frame):
    frame = frame.copy()
    xs, ys = frame[0::3], frame[1::3]
    valid = xs != 0
    if not valid.any():
        return frame
    cx, cy = xs[0], ys[0]
    scale = max(xs[valid].max() - xs[valid].min(),
                ys[valid].max() - ys[valid].min())
    if scale == 0: scale = 1
    for i in range(0, len(frame), 3):
        frame[i]     = (frame[i]     - cx) / scale
        frame[i + 1] = (frame[i + 1] - cy) / scale
    return frame

def normalize_pose(frame):
    frame = frame.copy()
    xs, ys = frame[0::3], frame[1::3]
    valid = xs != 0
    if not valid.any():
        return frame
    cx, cy = xs[0], ys[0]
    scale = max(xs[valid].max() - xs[valid].min(),
                ys[valid].max() - ys[valid].min())
    if scale == 0: scale = 1
    for i in range(0, len(frame), 3):
        frame[i]     = (frame[i]     - cx) / scale
        frame[i + 1] = (frame[i + 1] - cy) / scale
    return frame

def normalize_sequence(seq):
    """Normalize a (SEQUENCE_LENGTH, FEATURE_SIZE) array per-frame."""
    return np.array([
        np.concatenate([
            normalize_hand_pair(frame[:HAND_FEATURES]),
            normalize_face(frame[HAND_FEATURES:HAND_FEATURES + FACE_FEATURES]),
            normalize_pose(frame[HAND_FEATURES + FACE_FEATURES:])
        ])
        for frame in seq
    ], dtype=np.float32)

# ==============================
# MIRROR (left ↔ right hand swap)
# ==============================
def mirror_sequence(sequence):
    """
    Horizontal flip: swap left hand (0:63) ↔ right hand (63:126),
    negate x coordinates across all landmarks, swap shoulder/elbow/wrist pairs
    in pose (indices 0-2 ↔ 3-5 in the 6-point pose block).
    """
    new_seq = []
    for frame in sequence:
        f = frame.copy()

        # Flip x for all landmarks
        for i in range(0, FEATURE_SIZE, 3):
            if f[i] != 0:
                f[i] = 1.0 - f[i]

        # Swap left hand block ↔ right hand block
        left  = f[0:63].copy()
        right = f[63:126].copy()
        f[0:63]   = right
        f[63:126] = left

        # Swap pose pairs: left shoulder↔right shoulder, elbow↔elbow, wrist↔wrist
        # POSE_IDX = [11,13,15, 12,14,16] → stored as 6 × 3 = 18 values
        # positions 0-8 = left side, 9-17 = right side
        pose_start = HAND_FEATURES + FACE_FEATURES
        left_pose  = f[pose_start:pose_start + 9].copy()
        right_pose = f[pose_start + 9:pose_start + 18].copy()
        f[pose_start:pose_start + 9]      = right_pose
        f[pose_start + 9:pose_start + 18] = left_pose

        new_seq.append(f)
    return np.array(new_seq, dtype=np.float32)

# ==============================
# AUGMENTATION FUNCTIONS
# ==============================

def spatial_augment(sequence):
    """Global scale + rotation + translation applied to all landmarks."""
    scale = random.uniform(*AUG_SCALE_RANGE)
    angle = random.uniform(*AUG_ROT_RANGE)
    tx    = random.uniform(*AUG_TRANS_RANGE)
    ty    = random.uniform(*AUG_TRANS_RANGE)
    theta = math.radians(angle)
    cos_t, sin_t = math.cos(theta), math.sin(theta)

    new_seq = []
    for frame in sequence:
        f = frame.copy()
        for i in range(0, FEATURE_SIZE, 3):
            x, y, z = f[i], f[i+1], f[i+2]
            x *= scale; y *= scale
            f[i]     = x * cos_t - y * sin_t + tx
            f[i + 1] = x * sin_t + y * cos_t + ty
            f[i + 2] = z
        new_seq.append(f)
    return np.array(new_seq, dtype=np.float32)


def noise_augment(sequence):
    """Add gaussian noise to all keypoint values."""
    noise = np.random.normal(0, AUG_NOISE_STD, sequence.shape).astype(np.float32)
    return sequence + noise


def hand_scale_augment(sequence):
    """
    Independently scale each hand around its own wrist (landmark 0).
    Simulates natural variation in hand size between different users.
    """
    left_scale  = random.uniform(*AUG_HAND_SCALE)
    right_scale = random.uniform(*AUG_HAND_SCALE)
    new_seq = []
    for frame in sequence:
        f = frame.copy()
        # Left hand: wrist at index 0,1
        if f[0] != 0:
            cx, cy = f[0], f[1]
            for i in range(0, 63, 3):
                f[i]     = cx + (f[i]     - cx) * left_scale
                f[i + 1] = cy + (f[i + 1] - cy) * left_scale
        # Right hand: wrist at index 63,64
        if f[63] != 0:
            cx, cy = f[63], f[64]
            for i in range(63, 126, 3):
                f[i]     = cx + (f[i]     - cx) * right_scale
                f[i + 1] = cy + (f[i + 1] - cy) * right_scale
        new_seq.append(f)
    return np.array(new_seq, dtype=np.float32)


def perspective_augment(sequence):
    """
    X-shear to simulate camera viewing angle or user leaning.
    x += shear * y shifts landmarks horizontally proportional to height.
    """
    shear = random.uniform(*AUG_SHEAR_RANGE)
    new_seq = []
    for frame in sequence:
        f = frame.copy()
        for i in range(0, FEATURE_SIZE, 3):
            if f[i] != 0:
                f[i] = f[i] + shear * f[i + 1]
        new_seq.append(f)
    return np.array(new_seq, dtype=np.float32)


def body_shift_augment(sequence):
    """
    Apply slightly different translation offsets per body region.
    Simulates the user standing off-centre in the frame.
    """
    hand_tx = random.uniform(*AUG_TRANS_RANGE)
    hand_ty = random.uniform(*AUG_TRANS_RANGE)
    # Face and pose drift slightly relative to hands
    face_tx = hand_tx + random.uniform(-0.02, 0.02)
    face_ty = hand_ty + random.uniform(-0.02, 0.02)

    new_seq = []
    for frame in sequence:
        f = frame.copy()
        # Hands (0:126)
        for i in range(0, 126, 3):
            if f[i] != 0:
                f[i]     += hand_tx
                f[i + 1] += hand_ty
        # Face (126:186)
        for i in range(126, 186, 3):
            if f[i] != 0:
                f[i]     += face_tx
                f[i + 1] += face_ty
        # Pose (186:204) — follows hands
        for i in range(186, 204, 3):
            if f[i] != 0:
                f[i]     += hand_tx
                f[i + 1] += hand_ty
        new_seq.append(f)
    return np.array(new_seq, dtype=np.float32)


def time_warp_augment(sequence):
    """
    Stretch or compress the sequence in time to simulate signing speed variation.
    Resamples to a warped length then back to SEQUENCE_LENGTH via linear interp.
    """
    factor  = random.uniform(*AUG_WARP_RANGE)
    n       = len(sequence)
    new_len = max(2, int(n * factor))

    # Resample to new_len
    src_idx   = np.linspace(0, n - 1, new_len)
    resampled = np.array([
        sequence[int(i)] * (1 - (i % 1)) +
        sequence[min(int(i) + 1, n - 1)] * (i % 1)
        for i in src_idx
    ], dtype=np.float32)

    # Resize back to SEQUENCE_LENGTH
    tgt_idx = np.linspace(0, len(resampled) - 1, SEQUENCE_LENGTH)
    return np.array([
        resampled[int(i)] * (1 - (i % 1)) +
        resampled[min(int(i) + 1, len(resampled) - 1)] * (i % 1)
        for i in tgt_idx
    ], dtype=np.float32)


def smooth_frame_drop(sequence):
    """
    Drop 0–2 random frames and fill gaps via linear interpolation between
    neighbours instead of repeating the last frame (smoother motion).
    """
    seq        = list(sequence)
    drop_count = random.randint(0, 2)

    for _ in range(drop_count):
        if len(seq) > 2:
            idx      = random.randint(1, len(seq) - 2)
            interp   = ((seq[idx - 1] + seq[idx + 1]) / 2.0).astype(np.float32)
            seq[idx] = interp

    # Pad if needed
    while len(seq) < SEQUENCE_LENGTH:
        if len(seq) > 1:
            interp = ((seq[-1] + seq[-2]) / 2.0).astype(np.float32)
        else:
            interp = seq[-1].copy()
        seq.append(interp)

    return np.array(seq[:SEQUENCE_LENGTH], dtype=np.float32)


def keypoint_dropout_augment(sequence):
    """
    Randomly zero out entire hands or individual face/pose landmarks.
    Trains the model to handle partial MediaPipe detections (occlusion).
    """
    new_seq = []
    for frame in sequence:
        f = frame.copy()
        # Randomly drop an entire hand
        if random.random() < AUG_KP_DROP_P:
            f[0:63] = 0.0      # drop left hand
        if random.random() < AUG_KP_DROP_P:
            f[63:126] = 0.0    # drop right hand
        # Randomly drop individual face/pose landmarks
        for i in range(126, FEATURE_SIZE, 3):
            if random.random() < AUG_KP_DROP_P:
                f[i] = f[i+1] = f[i+2] = 0.0
        new_seq.append(f)
    return np.array(new_seq, dtype=np.float32)


def mixup_augment(seq_a, seq_b):
    """
    Blend two sequences of the same class together.
    Forces the model to learn smooth interpolations between signing styles.
    """
    lam = np.random.beta(AUG_MIXUP_ALPHA, AUG_MIXUP_ALPHA)
    return (lam * seq_a + (1 - lam) * seq_b).astype(np.float32)


def augment_sequence(sequence, same_class_pool=None):
    """
    Full augmentation pipeline. Applies each transform with appropriate
    probability. same_class_pool is a list of sequences from the same class
    used for mixup (optional).
    """
    seq = sequence.copy()

    # Always apply base spatial + noise
    seq = spatial_augment(seq)
    seq = noise_augment(seq)

    # Body-level variation
    seq = hand_scale_augment(seq)
    seq = body_shift_augment(seq)

    # Perspective simulation (always)
    seq = perspective_augment(seq)

    # Temporal variation
    seq = smooth_frame_drop(seq)
    if random.random() < AUG_WARP_PROB:
        seq = time_warp_augment(seq)

    # Occlusion simulation
    seq = keypoint_dropout_augment(seq)

    # Mirror (50% probability)
    if random.random() < AUG_MIRROR_PROB:
        seq = mirror_sequence(seq)

    # Mixup with same-class sample (30% probability, only if pool available)
    if same_class_pool is not None and random.random() < AUG_MIXUP_PROB:
        partner = random.choice(same_class_pool)
        seq = mixup_augment(seq, partner)

    return seq

# ==============================
# LOAD DATASET
# ==============================
def load_dataset(data_path):
    X, y = [], []

    for category in sorted(os.listdir(data_path)):
        cat_path = os.path.join(data_path, category)
        if not os.path.isdir(cat_path):
            continue

        for label in sorted(os.listdir(cat_path)):
            label_folder = os.path.join(cat_path, label)
            if not os.path.isdir(label_folder):
                continue

            files = [f for f in os.listdir(label_folder) if f.endswith(".npy")]
            if not files:
                continue

            for file in files:
                seq = np.load(os.path.join(label_folder, file))

                if seq.shape != (SEQUENCE_LENGTH, FEATURE_SIZE):
                    print(f"  ⚠️  Skipping {file} — shape {seq.shape}")
                    continue

                nonzero = np.count_nonzero(seq) / seq.size
                if nonzero < MIN_NONZERO:
                    print(f"  ⚠️  Skipping {file} — low quality {nonzero:.0%}")
                    continue

                seq_norm = normalize_sequence(seq)
                X.append(seq_norm)
                y.append(label)

    return np.array(X, dtype=np.float32), np.array(y)

# ==============================
# MAIN
# ==============================
print("=" * 60)
print("  BIM Sign Language — Model Training  (enhanced augmentation)")
print("=" * 60)

print("\n📂 Loading dataset...")
X, y = load_dataset(DATA_PATH)
labels = sorted(list(set(y)))
print(f"   Raw sequences : {len(X)}")
print(f"   Classes       : {len(labels)} — {labels}")
print(f"   Class counts  : {dict(Counter(y))}")

if len(X) == 0:
    raise RuntimeError(f"No data found in {DATA_PATH}. Run 00_data_collection.py first.")

# ── Encode labels ──
le    = LabelEncoder()
y_enc = le.fit_transform(y)

# ── Split FIRST, augment second (no leakage) ──
print("\n✂️  Splitting dataset (before augmentation)...")
X_train_raw, X_test, y_train_raw, y_test_enc = train_test_split(
    X, y_enc, test_size=TEST_SIZE, random_state=42, stratify=y_enc
)
print(f"   Train (raw) : {len(X_train_raw)} | Test : {len(X_test)}")

# ── Build per-class pool for mixup ──
class_pool: dict[int, list] = defaultdict(list)
for seq, lbl in zip(X_train_raw, y_train_raw):
    class_pool[int(lbl)].append(seq)

# ── Augment training set ──
print(f"\n🔀 Augmenting training set (×{AUGMENTATION_FACTOR} copies per sample)...")
X_train_aug = list(X_train_raw)
y_train_aug = list(y_train_raw)

for seq, lbl in zip(X_train_raw, y_train_raw):
    pool = class_pool[int(lbl)]
    for _ in range(AUGMENTATION_FACTOR):
        X_train_aug.append(augment_sequence(seq, same_class_pool=pool))
        y_train_aug.append(lbl)

X_train     = np.array(X_train_aug, dtype=np.float32)
y_train_enc = np.array(y_train_aug)

# Shuffle
perm        = np.random.permutation(len(X_train))
X_train     = X_train[perm]
y_train_enc = y_train_enc[perm]

print(f"   Train (augmented) : {len(X_train)}")
print(f"   Augmentation breakdown per original sample:")
print(f"     • spatial + noise        : always")
print(f"     • hand scale + body shift: always")
print(f"     • perspective            : always")
print(f"     • smooth frame drop      : always")
print(f"     • time warp              : {AUG_WARP_PROB*100:.0f}% probability")
print(f"     • keypoint dropout       : always (p={AUG_KP_DROP_P})")
print(f"     • mirror                 : {AUG_MIRROR_PROB*100:.0f}% probability")
print(f"     • mixup                  : {AUG_MIXUP_PROB*100:.0f}% probability")

# ── One-hot encode ──
num_classes  = len(labels)
y_train_oh   = tf.keras.utils.to_categorical(y_train_enc, num_classes)
y_test_oh    = tf.keras.utils.to_categorical(y_test_enc,  num_classes)

# ── Save label encoder + class names ──
with open(ENCODER_PATH, "wb") as f:
    pickle.dump(le, f)
with open(LABEL_PATH, "w") as f:
    json.dump(labels, f, indent=2)
print(f"\n💾 Label encoder → {ENCODER_PATH}")
print(f"💾 Labels JSON   → {LABEL_PATH}")

# ==============================
# MODEL
# ==============================
print(f"\n🏗️  Building model  (input: {SEQUENCE_LENGTH} × {FEATURE_SIZE}, classes: {num_classes})...")

model = Sequential([
    # ── Block 1: local pattern detection ──
    Conv1D(64, kernel_size=5, activation='relu', padding='same',
           input_shape=(SEQUENCE_LENGTH, FEATURE_SIZE)),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    # ── Block 2: higher-level feature extraction ──
    Conv1D(128, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    Dropout(0.3),

    # ── Block 3: temporal sequence modelling ──
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.4),

    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.4),

    # ── Classifier ──
    Dense(128, activation='relu',
          kernel_regularizer=regularizers.l2(1e-4)),
    Dropout(0.4),

    Dense(64, activation='relu',
          kernel_regularizer=regularizers.l2(1e-4)),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ==============================
# CALLBACKS
# ==============================
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),
]

# ==============================
# TRAIN
# ==============================
print(f"\n🏋️  Training  (epochs={EPOCHS}, batch={BATCH_SIZE})...")
history = model.fit(
    X_train, y_train_oh,
    validation_split=0.1,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

with open(HISTORY_PATH, "wb") as f:
    pickle.dump(history.history, f)

# ==============================
# EVALUATE
# ==============================
loss, acc = model.evaluate(X_test, y_test_oh, verbose=0)
print(f"\n📊 Test Accuracy : {acc * 100:.2f}%")
print(f"   Test Loss     : {loss:.4f}")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print("\n📋 Classification Report:")
print(classification_report(y_test_enc, y_pred, target_names=labels))
print("Confusion Matrix:")
print(confusion_matrix(y_test_enc, y_pred))

# Per-class accuracy summary
print("\n📈 Per-class accuracy:")
for i, lbl in enumerate(labels):
    mask     = y_test_enc == i
    if mask.sum() == 0: continue
    cls_acc  = (y_pred[mask] == i).mean() * 100
    n        = mask.sum()
    status   = "✅" if cls_acc >= 80 else "⚠️ " if cls_acc >= 60 else "❌"
    print(f"  {status}  {lbl:<20} {cls_acc:5.1f}%  ({n} test samples)")

# ── Save Keras model ──
model.save(MODEL_SAVE_PATH)
print(f"\n💾 Keras model saved → {MODEL_SAVE_PATH}")

# ==============================
# TFLITE EXPORT
# ==============================
# print("\n📦 Converting to TFLite...")
# converter              = tf.lite.TFLiteConverter.from_keras_model(model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model           = converter.convert()

# with open(TFLITE_PATH, "wb") as f:
#     f.write(tflite_model)

# size_kb = os.path.getsize(TFLITE_PATH) / 1024
# print(f"💾 TFLite model saved → {TFLITE_PATH}  ({size_kb:.1f} KB)")


import os
import shutil

# Source files
MODEL_PATH = "bim_model/handface_pose_cnn_lstm.h5"
LABEL_PATH = "bim_model/labels.json"

# Destination folders
MODEL_DEST_FOLDER = "deploy/model/"
LABEL_DEST_FOLDER = "deploy/labels/"

# Ensure destination folders exist
os.makedirs(MODEL_DEST_FOLDER, exist_ok=True)
os.makedirs(LABEL_DEST_FOLDER, exist_ok=True)

# Destination paths
model_dest = os.path.join(MODEL_DEST_FOLDER, os.path.basename(MODEL_PATH))
label_dest = os.path.join(LABEL_DEST_FOLDER, os.path.basename(LABEL_PATH))

# Copy files
shutil.copy2(MODEL_PATH, model_dest)
shutil.copy2(LABEL_PATH, label_dest)

print(f"Model copied to: {model_dest}")
print(f"Labels copied to: {label_dest}")

print("\n🎉 Training complete!")

  BIM Sign Language — Model Training  (enhanced augmentation)

📂 Loading dataset...
   Raw sequences : 480
   Classes       : 4 — ['A', 'B', 'C', 'E']
   Class counts  : {'A': 120, 'B': 120, 'C': 120, 'E': 120}

✂️  Splitting dataset (before augmentation)...
   Train (raw) : 384 | Test : 96

🔀 Augmenting training set (×5 copies per sample)...
   Train (augmented) : 2304
   Augmentation breakdown per original sample:
     • spatial + noise        : always
     • hand scale + body shift: always
     • perspective            : always
     • smooth frame drop      : always
     • time warp              : 40% probability
     • keypoint dropout       : always (p=0.12)
     • mirror                 : 50% probability
     • mixup                  : 30% probability

💾 Label encoder → bim_model/label_encoder.pkl
💾 Labels JSON   → bim_model/labels.json

🏗️  Building model  (input: 8 × 204, classes: 4)...
Model: "sequential_9"
_________________________________________________________________
 Lay